# Revision controls for the head-set result (Colab)
Follow-up to `revision_experiments.ipynb`. That run found that the top patching heads are jointly sufficient
(top-10 restore 94% of the logit gap) and that zeroing the top-40 drops AUC to chance. Three controls decide
what the paper may claim:

| § | Control | Question |
|---|---|---|
| 6–7 | **Mean-ablation** vs zero-ablation | Is the collapse an off-distribution artefact of zeroing? |
| 5–7 | **Off-task readouts** (SST-2 sentiment, AG News sports) | Do the heads break only suicidality, or every yes/no judgement? |
| 8 | **Heads → layer-16 direction** (knockout and patching) | Do these heads write the direction the paper ablates? |

Same Drive layout and bundle as before (`MyDrive/MI on suicide/`). It reads `direction_L16.npy` and
`joint_topk_heads.json` from `Dataset/Outputs/revision_colab/` (written by the first notebook) and writes to
`revision_colab/controls/` and `Dataset/Outputs/revision_controls.zip`. A100: about 20 minutes. Needs internet for
SST-2 and AG News (Hugging Face datasets).

In [ ]:
# 0.1  Runtime + dependencies
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.56.2",
                    "accelerate>=0.33", "datasets>=2.19", "datasketch"], check=True)

In [ ]:
# 0.2  Paths, Drive, Hugging Face login, fixed settings
from pathlib import Path
import json, time, zipfile, copy
if IN_COLAB:
    from google.colab import drive, userdata
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    # Same Drive project as llama_all_phases_colab.ipynb
    DRIVE_PROJECT = Path("/content/drive/MyDrive/MI on suicide")
    assert DRIVE_PROJECT.exists(), f"{DRIVE_PROJECT} not found"
    BASE = DRIVE_PROJECT / "Dataset" / "Outputs"
    WORK = Path("/content/work")
    # Reuse the Llama weights already cached on Drive (no 16 GB re-download).
    # Must be set before transformers / huggingface_hub are imported.
    os.environ["HF_HOME"] = str(DRIVE_PROJECT / "hf_cache")
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(DRIVE_PROJECT / "hf_cache" / "hub")
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    BUNDLE_CANDIDATES = [DRIVE_PROJECT / "colab_data_bundle.zip",
                         DRIVE_PROJECT / "Dataset" / "Inputs" / "colab_data_bundle.zip",
                         DRIVE_PROJECT / "Dataset" / "colab_data_bundle.zip",
                         Path("/content/drive/MyDrive/colab_data_bundle.zip")]
else:
    BASE = Path(os.environ.get("REV_BASE", "."))
    WORK = Path(os.environ.get("REV_WORK", str(BASE / "work")))
    HF_TOKEN = os.environ.get("HF_TOKEN")
    BUNDLE_CANDIDATES = [BASE / "colab_data_bundle.zip"]

BUNDLE = next((p for p in BUNDLE_CANDIDATES if p.exists()), BUNDLE_CANDIDATES[0])
OUT = BASE / "revision_colab"
FIGS = OUT / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
if not (WORK / "data").exists():
    assert BUNDLE.exists(), "Upload colab_data_bundle.zip to one of: " + ", ".join(map(str, BUNDLE_CANDIDATES))
    print("unzipping bundle ..."); zipfile.ZipFile(BUNDLE).extractall(WORK)
DATA = WORK / "data" / "datasets"
RERUN = WORK / "results" / "sc2_binary"        # fp16 rerun at commit 0e9e39f2 (liu_headtohead_metallama_v2*)
OLD = WORK / "results" / "sc2_binary_fp16"     # the fp16 results the manuscript currently cites

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print("No HF_TOKEN secret. Run:  from huggingface_hub import notebook_login; notebook_login()")

MODEL_ID = os.environ.get("REV_MODEL_ID", "meta-llama/Llama-3.1-8B-Instruct")
REVISION = os.environ.get("REV_REVISION", "0e9e39f249a16976918f6564b8830bc894c89659")
SEED = 0
LAYER = 16          # hidden_states[16] == output of decoder layer 15
ABLATE_FROM = 15    # direction ablation hooks decoder layers 15..31
MAXLEN = 256        # body capped at MAXLEN-80 = 176 tokens (manuscript protocol)
TEMPLATE = ("Read the following text and determine if the person shows signs of "
            "suicidal ideation.\n\n"
            "Text: {text}\n\n"
            "Does this text show signs of suicidal ideation? Answer with Yes or No.\n"
            "Answer:")
PREFIX, SUFFIX = TEMPLATE.split("{text}")
print("OUT ->", OUT)

In [ ]:
# 0.3  Shared helpers (hashing, AUC, bootstrap, BH, JSON output)
import hashlib, re
import numpy as np
from scipy import stats
from scipy.stats import rankdata

ENV = {"seed": SEED, "model": MODEL_ID, "revision_requested": REVISION}

def h16(t):
    """Same hash as sc2_liu_headtohead_v2.py."""
    return hashlib.sha256(" ".join(t.split()).encode()).hexdigest()[:16]

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); q[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1.0)

def auc_cols(y, S):
    """ROC-AUC of every column of S (ties -> average rank). y: 0/1 vector."""
    S = np.asarray(S, float)
    S = S[:, None] if S.ndim == 1 else S
    r = rankdata(S, axis=0)
    n1 = int(y.sum()); n0 = len(y) - n1
    out = (r[y == 1].sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)
    return out

def gap_cols(y, S):
    S = np.asarray(S, float); S = S[:, None] if S.ndim == 1 else S
    return S[y == 1].mean(0) - S[y == 0].mean(0)

def strat_boot(y, B=2000, seed=SEED + 7):
    rng = np.random.default_rng(seed)
    P, N = np.where(y == 1)[0], np.where(y == 0)[0]
    for _ in range(B):
        yield np.concatenate([rng.choice(P, len(P)), rng.choice(N, len(N))])

def boot_p(d):
    """Two-sided bootstrap p-value for H0: effect = 0 (rows = resamples)."""
    return np.minimum(1.0, 2 * np.minimum((d <= 0).mean(0), (d >= 0).mean(0)))

def _js(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

def save_json(name, obj):
    path = OUT / name
    path.write_text(json.dumps({**obj, "env": ENV}, indent=2, default=_js))
    print("saved", path)

def fmt_ci(x, lo, hi, d=3):
    return f"{x:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"

In [ ]:
# 1.1  Suicide-Watch-232K draw (sc2_faithfulness.py / sc2_liu_headtohead_v2.py)
from datasets import load_from_disk
import csv
ds07 = load_from_disk(str(DATA / "07_Suicide_Watch_232K/hf_cache/train"))
T07, L07 = ds07["text"], ds07["label"]

def draw_main(seed=SEED):
    rng = np.random.default_rng(seed)
    sui = [t for t, l in zip(T07, L07) if l == "suicide" and t and len(t) > 80]
    non = [t for t, l in zip(T07, L07) if l == "non-suicide" and t and len(t) > 80]
    rng.shuffle(sui); rng.shuffle(non)
    return sui, non, rng

SUI, NON, _rng = draw_main()
RNG_AFTER_SHUFFLE = copy.deepcopy(_rng)   # the random-control direction is the next draw from this
SUI_TR, NON_TR = SUI[:100], NON[:100]          # direction fit
SUI_TE, NON_TE = SUI[100:300], NON[100:300]    # held-out test; first 100/class == manuscript test set

abl_rows = list(csv.DictReader(open(RERUN / "liu_headtohead_metallama_v2_ablation_examples.csv")))
assert [r["hash"] for r in abl_rows] == [h16(t) for t in SUI_TE + NON_TE], "test draw differs from the rerun!"
print(f"pools: {len(SUI)} suicide / {len(NON)} non-suicide;  test set matches the fp16 rerun (400/400 hashes)")

In [ ]:
# 1.3  Tokenizer + prompt builder (identical to the manuscript scripts)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
YES = tok.encode(" Yes", add_special_tokens=False)[0]
NO = tok.encode(" No", add_special_tokens=False)[0]

def chat_ids(text, maxlen=MAXLEN):
    body = tok(text, add_special_tokens=False)["input_ids"]
    truncated = len(body) > maxlen - 80
    content = PREFIX + tok.decode(body[: maxlen - 80]) + SUFFIX
    s = tok.apply_chat_template([{"role": "user", "content": content}],
                                add_generation_prompt=True, tokenize=False)
    return tok(s, add_special_tokens=False, return_tensors="pt")["input_ids"], truncated

def body_len(text):
    return len(tok(text, add_special_tokens=False)["input_ids"])

ids0, tr0 = chat_ids(SUI_TE[0])
assert ids0.shape[1] == int(abl_rows[0]["n_tokens"]), "prompt length differs from the rerun"
print("prompt builder matches the rerun (token count check)")

## 2. Model, direction and head sets (GPU)
Loads the pinned fp16 model, the layer-16 direction saved by the first notebook (`direction_L16.npy`, refit
here if missing), and the patching head ranking (`joint_topk_heads.json`). Random controls are the same
layer-matched sets as before (same seed and call order).

In [ ]:
import torch, transformers
from transformers import AutoModelForCausalLM
DEV = "cuda"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float16,
                                             device_map={"": 0})
model.eval()
LAYERS = model.model.layers
NL, NH = len(LAYERS), model.config.num_attention_heads
HD = getattr(model.config, "head_dim", None) or model.config.hidden_size // NH
DM = model.config.hidden_size
ENV.update(revision_resolved=getattr(model.config, "_commit_hash", None), precision="fp16",
           transformers=transformers.__version__, torch=torch.__version__, gpu=torch.cuda.get_device_name(0))
CTRL = OUT / "controls"; (CTRL / "figs").mkdir(parents=True, exist_ok=True)

@torch.inference_mode()
def hidden16(ids):
    out = model(ids.to(DEV), output_hidden_states=True, logits_to_keep=1)
    return out.hidden_states[LAYER][0, -1].float().cpu().numpy()

if (OUT / "direction_L16.npy").exists():
    DIR = np.load(OUT / "direction_L16.npy")
else:
    Htr = np.stack([hidden16(chat_ids(t)[0]) for t in SUI_TR + NON_TR])
    DIR = Htr[:100].mean(0) - Htr[100:].mean(0); DIR /= np.linalg.norm(DIR)
DIR_T = torch.tensor(DIR, device=DEV, dtype=torch.float32)

def hn(k): return f"L{int(k) // NH}H{int(k) % NH}"
def hid(name): L, h = map(int, name[1:].split("H")); return L * NH + h

jt = OUT / "joint_topk_heads.json"
if jt.exists():
    RANK = [hid(n) for n in json.loads(jt.read_text())["heads_full_rank"]]
else:
    _z = np.load(RERUN / "liu_headtohead_metallama_v2_ie.npz")
    RANK = list(np.argsort(-_z["ie"][_z["primary"].astype(bool)].mean(0).ravel())[:40])
KS_ALL = [1, 3, 5, 10, 20, 40]; KS = [5, 10, 20, 40]; N_RAND = 10
rr = np.random.default_rng(SEED + 23)
def layer_matched(heads):
    chosen = set(int(x) for x in heads); out = []
    for k in heads:
        L = int(k) // NH
        opts = [L * NH + h for h in range(NH) if L * NH + h not in chosen]
        c = int(rr.choice(opts)); chosen.add(c); out.append(c)
    return out
RAND_SETS = {k: [layer_matched(RANK[:k]) for _ in range(N_RAND)] for k in KS_ALL}   # same sets as notebook 1
SETS = [("top", k, RANK[:k]) for k in KS] + [("rand", k, s) for k in KS for s in RAND_SETS[k]]
print("top-40 heads:", [hn(k) for k in RANK])
print("top-k heads in layers <= 15 (the only ones that can change hidden_states[16]):",
      {k: sum(int(h) // NH <= 15 for h in RANK[:k]) for k in KS})

## 3. Per-head mean activations (for mean-ablation)
Mean of each head's output (`o_proj` input slice) over every token position of the 200 direction-training
prompts (class-balanced, disjoint from all evaluation posts). Mean-ablation replaces a head's output with this
constant instead of zero, which keeps the residual stream on-distribution.

In [ ]:
@torch.inference_mode()
def head_means(texts):
    sums = torch.zeros(NL, NH * HD, device=DEV, dtype=torch.float64); cnt = 0
    def mk(L):
        def pre(m, inp):
            sums[L] += inp[0][0].double().sum(0)
        return pre
    hs = [LAYERS[L].self_attn.o_proj.register_forward_pre_hook(mk(L)) for L in range(NL)]
    try:
        for t in texts:
            ids = chat_ids(t)[0]; model(ids.to(DEV), logits_to_keep=1); cnt += ids.shape[1]
    finally:
        for x in hs: x.remove()
    return (sums / cnt).to(torch.float16)
MEANZ = head_means(SUI_TR + NON_TR)       # [NL, NH*HD]
print("mean activations ready", tuple(MEANZ.shape))

## 4. Batched intervention runner
One forward per chunk of head sets: each row zeroes, mean-ablates or clean-patches its own set of heads, and a
final unpatched row gives the reference. Returns the Yes−No logit difference **and** the final-token projection
of `hidden_states[16]` onto the suicidality direction for every row.

In [ ]:
def build_masks(head_sets, B):
    masks = {}
    for r, hs in enumerate(head_sets):
        for k in hs:
            L, h = divmod(int(k), NH)
            if L not in masks: masks[L] = torch.zeros(B, 1, NH * HD, dtype=torch.bool, device=DEV)
            masks[L][r, :, h * HD:(h + 1) * HD] = True
    return masks

@torch.inference_mode()
def run_sets(ids, head_sets, mode, clean_z=None, chunk=16):
    lds, projs = [], []
    for c0 in range(0, len(head_sets), chunk):
        sets = head_sets[c0:c0 + chunk] + [[]]
        masks = build_masks(sets, len(sets)); hooks = []; cap = {}
        for L, M in masks.items():
            def pre(module, inp, M=M, L=L):
                x = inp[0]
                if mode == "zero":   fill = torch.zeros((), dtype=x.dtype, device=x.device)
                elif mode == "mean": fill = MEANZ[L].to(x.dtype)[None, None]
                else:                fill = clean_z[L].to(x.dtype)[None]
                return (torch.where(M, fill, x),) + tuple(inp[1:])
            hooks.append(LAYERS[L].self_attn.o_proj.register_forward_pre_hook(pre))
        def grab(m, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            cap["p"] = (h[:, -1].float() @ DIR_T).cpu().numpy()
        hooks.append(LAYERS[LAYER - 1].register_forward_hook(grab))      # output of block 15 == hidden_states[16]
        try:
            ld = ld_batch(ids.repeat(len(sets), 1))
        finally:
            for x in hooks: x.remove()
        lds.append(ld); projs.append(cap["p"])
    ld_rows = np.concatenate([l[:-1] for l in lds]); pj_rows = np.concatenate([p[:-1] for p in projs])
    return ld_rows, pj_rows, float(np.mean([l[-1] for l in lds])), float(np.mean([p[-1] for p in projs]))

@torch.inference_mode()
def ld_batch(ids):
    lg = model(ids.to(DEV), logits_to_keep=1).logits[:, -1, :].float()
    return (lg[:, YES] - lg[:, NO]).cpu().numpy()

# consistency: hook-based projection == hidden_states[16] projection
_ids = chat_ids(SUI_TE[0])[0]
_, _, _, p_ref = run_sets(_ids, [[]], "zero")
assert abs(p_ref - float(hidden16(_ids) @ DIR)) < 0.05, "projection hook mismatch"
print("runner ok")

## 5. Off-task readouts (specificity control)
Two yes/no tasks the model should perform well and that have nothing to do with suicidality, posed in the same
chat format and read with the same Yes−No logit difference:
* **Sentiment** — SST-2 validation, "Is the sentiment of this text positive?"
* **Topic** — AG News test, "Is this news article about sports?"

100 per class each. If knocking out the head set destroys suicidality detection but leaves these intact, the
heads are task-specific; if all three collapse, the effect is generic damage.

In [ ]:
from datasets import load_dataset
def _load(names, split):
    for n in names:
        try:
            return load_dataset(n, split=split)
        except Exception as e:
            print(f"  {n}: {type(e).__name__}")
    raise RuntimeError(f"could not load any of {names}")

def make_builder(instruction, question):
    pre = f"Read the following text and determine {instruction}.\n\nText: "
    suf = f"\n\n{question} Answer with Yes or No.\nAnswer:"
    def build(text, maxlen=MAXLEN):
        body = tok(text, add_special_tokens=False)["input_ids"][: maxlen - 80]
        s = tok.apply_chat_template([{"role": "user", "content": pre + tok.decode(body) + suf}],
                                    add_generation_prompt=True, tokenize=False)
        return tok(s, add_special_tokens=False, return_tensors="pt")["input_ids"]
    return build

rng_t = np.random.default_rng(SEED + 41)
def pick(texts, labels, pos_fn, n=100, minlen=40):
    P = [t for t, l in zip(texts, labels) if pos_fn(l) and len(t) >= minlen]
    N = [t for t, l in zip(texts, labels) if not pos_fn(l) and len(t) >= minlen]
    P = [P[i] for i in rng_t.choice(len(P), n, replace=False)]; N = [N[i] for i in rng_t.choice(len(N), n, replace=False)]
    return P + N, np.array([1] * n + [0] * n)

sst = _load(["stanfordnlp/sst2", "glue/sst2", "sst2"], "validation")
sst_txt = sst["sentence"]; sst_lab = sst["label"]
agn = _load(["fancyzhx/ag_news", "ag_news"], "test")
TASKS = {
    "suicidality": {"ids": [chat_ids(t)[0] for t in SUI_TE[:100] + NON_TE[:100]],
                    "y": np.array([1] * 100 + [0] * 100)},
}
for name, (txt, lab, fn, ins, q) in {
        "sentiment": (sst_txt, sst_lab, lambda l: l == 1, "the sentiment of the text",
                      "Is the sentiment of this text positive?"),
        "topic_sports": (agn["text"], agn["label"], lambda l: l == 1, "the topic of the news article",
                         "Is this news article about sports?")}.items():
    texts, y = pick(txt, lab, fn); b = make_builder(ins, q)
    TASKS[name] = {"ids": [b(t) for t in texts], "y": y}
print({k: len(v["y"]) for k, v in TASKS.items()})

## 6. Run: zero- and mean-ablation of top-k vs layer-matched random head sets, on all three tasks
~1,200 posts × 2 modes × 3 batched forwards. About 10–15 min on an A100. Per-example results are saved.

In [ ]:
head_sets = [s for _, _, s in SETS]
RES = {}
t0 = time.time()
for task, T in TASKS.items():
    for mode in ("zero", "mean"):
        n = len(T["ids"]); LD = np.zeros((n, len(SETS))); PJ = np.zeros((n, len(SETS))); B = np.zeros((n, 2))
        for i, ids in enumerate(T["ids"]):
            LD[i], PJ[i], B[i, 0], B[i, 1] = run_sets(ids, head_sets, mode)
        RES[(task, mode)] = {"ld": LD, "proj": PJ, "base_ld": B[:, 0], "base_proj": B[:, 1], "y": T["y"]}
        print(f"  {task:13s} {mode}: done ({time.time() - t0:.0f}s)")
np.savez_compressed(CTRL / "controls_per_example.npz",
                    **{f"{t}__{m}__{k}": v for (t, m), d in RES.items() for k, v in d.items()},
                    sets=np.array([f"{a}{k}" for a, k, _ in SETS]))

## 7. Results: specificity (does it break only suicidality?) and zero vs mean ablation

In [ ]:
def summarize(d, key="ld", B=1000):
    y = d["y"]; S = np.concatenate([d["base_" + key][:, None], d[key]], 1)
    a, g = auc_cols(y, S), gap_cols(y, S)
    bt = np.array([auc_cols(y[ix], S[ix]) for ix in strat_boot(y, B)])
    return a, g, bt

SPEC = {}
for (task, mode), d in RES.items():
    a, g, bt = summarize(d)
    rows = {}
    for k in KS:
        ti = 1 + [i for i, (kind, kk, _) in enumerate(SETS) if kind == "top" and kk == k][0]
        ri = 1 + np.array([i for i, (kind, kk, _) in enumerate(SETS) if kind == "rand" and kk == k])
        drop = bt[:, 0] - bt[:, ti]
        rows[k] = {"auc_top": a[ti], "auc_drop_top_ci95": ci(drop), "gap_top": g[ti],
                   "auc_rand_mean": a[ri].mean(), "auc_rand_range": [a[ri].min(), a[ri].max()],
                   "gap_rand_mean": g[ri].mean()}
    SPEC[f"{task}/{mode}"] = {"baseline_auc": a[0], "baseline_gap": g[0], "by_k": rows}

print(f"{'task / mode':24s} {'base AUC':>8s} " + " ".join(f"{'top-' + str(k) + ' (rand)':>20s}" for k in KS))
for key, s in SPEC.items():
    print(f"{key:24s} {s['baseline_auc']:8.3f} " + " ".join(
        f"{s['by_k'][k]['auc_top']:8.3f} ({s['by_k'][k]['auc_rand_mean']:.3f})    " for k in KS))
print("\nLogit-difference gap (suicide-relevant scale):")
for key, s in SPEC.items():
    print(f"{key:24s} {s['baseline_gap']:8.2f} " + " ".join(
        f"{s['by_k'][k]['gap_top']:8.2f} ({s['by_k'][k]['gap_rand_mean']:.2f})    " for k in KS))

# selectivity index: AUC lost on suicidality minus AUC lost on the off-task readouts, top-40 mean-ablation
for mode in ("zero", "mean"):
    base = {t: SPEC[f"{t}/{mode}"]["baseline_auc"] for t in TASKS}
    lost = {t: base[t] - SPEC[f"{t}/{mode}"]["by_k"][40]["auc_top"] for t in TASKS}
    print(f"\n{mode}-ablation of top-40: AUC lost = " + ", ".join(f"{t} {v:.3f}" for t, v in lost.items()))
save_json("controls/specificity_mean_ablation.json", SPEC)

## 8. Do the heads write the layer-16 direction?
(a) **Knockout**: under zero/mean ablation of the top-k heads, how much of the class separation of the
final-token projection onto the direction at `hidden_states[16]` survives (projection AUC and gap)? Only heads in
layers ≤ 15 can affect it, so the sets are also evaluated restricted to those heads.
(b) **Patching**: on the 60 held-out pairs, restoring the clean outputs of the top-k heads — what fraction of the
clean–corrupt *projection* difference is recovered (projection-IE), next to the logit IE?

In [ ]:
WRITE = {"knockout": {}, "patching": {}}
for mode in ("zero", "mean"):
    d = RES[("suicidality", mode)]
    a, g, bt = summarize(d, key="proj")
    out = {"baseline_proj_auc": a[0], "baseline_proj_gap": g[0], "by_k": {}}
    for k in KS:
        ti = 1 + [i for i, (kind, kk, _) in enumerate(SETS) if kind == "top" and kk == k][0]
        ri = 1 + np.array([i for i, (kind, kk, _) in enumerate(SETS) if kind == "rand" and kk == k])
        out["by_k"][k] = {"proj_auc_top": a[ti], "proj_auc_drop_ci95": ci(bt[:, 0] - bt[:, ti]),
                          "proj_gap_top": g[ti], "proj_gap_frac_remaining": g[ti] / g[0],
                          "proj_auc_rand_mean": a[ri].mean(), "proj_gap_rand_frac": (g[ri] / g[0]).mean()}
    WRITE["knockout"][mode] = out
    print(f"{mode}-ablation: L16 projection AUC {a[0]:.3f}, gap {g[0]:.2f} -> " + "; ".join(
        f"top-{k}: AUC {out['by_k'][k]['proj_auc_top']:.3f}, gap {100 * out['by_k'][k]['proj_gap_frac_remaining']:.0f}% "
        f"(random {100 * out['by_k'][k]['proj_gap_rand_frac']:.0f}%)" for k in KS))

pair_rows = list(csv.DictReader(open(RERUN / "liu_headtohead_metallama_v2_patching_pairs.csv")))
_npz = np.load(RERUN / "liu_headtohead_metallama_v2_ie.npz"); PRIM = _npz["primary"].astype(bool)
look_s = {h16(t): t for t in SUI[300:6300]}; look_n = {h16(t): t for t in NON[300:6300]}
PAIRS_P = [(look_s[r["hash_suicide"]], look_n[r["hash_non_suicide"]]) for r, k in zip(pair_rows, PRIM) if k]

@torch.inference_mode()
def cache_clean(ids):
    store, cap = {}, {}
    def mk(L):
        def pre(m, inp): store[L] = inp[0][0].detach().clone()
        return pre
    hs = [LAYERS[L].self_attn.o_proj.register_forward_pre_hook(mk(L)) for L in range(NL)]
    def grab(m, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        cap["p"] = float(h[0, -1].float() @ DIR_T)
    hs.append(LAYERS[LAYER - 1].register_forward_hook(grab))
    try:
        ld = ld_batch(ids)[0]
    finally:
        for x in hs: x.remove()
    return ld, cap["p"], store

IE_LD = np.zeros((len(PAIRS_P), len(SETS))); IE_PJ = np.zeros((len(PAIRS_P), len(SETS)))
for j, (ts, tn) in enumerate(PAIRS_P):
    ld_c, pj_c, z = cache_clean(chat_ids(ts)[0])
    ld, pj, ld_n, pj_n = run_sets(chat_ids(tn)[0], head_sets, "clean", clean_z=z)
    IE_LD[j] = (ld - ld_n) / (ld_c - ld_n); IE_PJ[j] = (pj - pj_n) / (pj_c - pj_n)
for k in KS:
    ti = [i for i, (kind, kk, _) in enumerate(SETS) if kind == "top" and kk == k][0]
    ri = np.array([i for i, (kind, kk, _) in enumerate(SETS) if kind == "rand" and kk == k])
    bi = np.random.default_rng(SEED + 51).integers(0, len(PAIRS_P), (2000, len(PAIRS_P)))
    WRITE["patching"][k] = {"ld_ie_top": IE_LD[:, ti].mean(), "proj_ie_top": IE_PJ[:, ti].mean(),
                            "proj_ie_top_ci95": ci(np.array([IE_PJ[b, ti].mean() for b in bi])),
                            "median_proj_ie_top": np.median(IE_PJ[:, ti]),
                            "proj_ie_rand_mean": IE_PJ[:, ri].mean(), "ld_ie_rand_mean": IE_LD[:, ri].mean()}
    w = WRITE["patching"][k]
    print(f"patch top-{k}: logit IE {w['ld_ie_top']:.3f}, L16-projection IE {w['proj_ie_top']:.3f} "
          f"[{w['proj_ie_top_ci95'][0]:.3f}, {w['proj_ie_top_ci95'][1]:.3f}] (median {w['median_proj_ie_top']:.3f}); "
          f"random: logit {w['ld_ie_rand_mean']:.3f}, projection {w['proj_ie_rand_mean']:.3f}")
save_json("controls/heads_write_direction.json", WRITE)

## 9. Figure + verdict

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
cols = {"suicidality": "#b2182b", "sentiment": "#2166ac", "topic_sports": "#4d9221"}
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for i, mode in enumerate(("zero", "mean")):
    for t, c in cols.items():
        s = SPEC[f"{t}/{mode}"]
        top = [s["baseline_auc"]] + [s["by_k"][k]["auc_top"] for k in KS]
        lo = [s["baseline_auc"]] + [s["by_k"][k]["auc_rand_range"][0] for k in KS]
        hi = [s["baseline_auc"]] + [s["by_k"][k]["auc_rand_range"][1] for k in KS]
        xs = [0] + KS
        ax[i].plot(xs, top, "o-", c=c, ms=4, label=t)
        ax[i].fill_between(xs, lo, hi, color=c, alpha=0.15, lw=0)
    ax[i].axhline(0.5, c="0.6", lw=0.6, ls=":"); ax[i].set_ylim(0.45, 1.01)
    ax[i].set_xlabel("top-k heads ablated (bands: layer-matched random)"); ax[i].set_ylabel("AUC")
    ax[i].set_title(f"{'ab'[i]}  {mode}-ablation", loc="left"); ax[i].legend(frameon=False, fontsize=7)
for mode, ls in (("zero", "--"), ("mean", "-")):
    ko = WRITE["knockout"][mode]["by_k"]
    ax[2].plot(KS, [100 * ko[k]["proj_gap_frac_remaining"] for k in KS], "o" + ls, c="#762a83", ms=4,
               label=f"knockout ({mode}): % L16 separation left")
ax[2].plot(KS, [100 * WRITE["patching"][k]["proj_ie_top"] for k in KS], "s-", c="#e08214", ms=4,
           label="patching: % L16 projection restored")
ax[2].plot(KS, [100 * WRITE["patching"][k]["ld_ie_top"] for k in KS], "s:", c="0.4", ms=4,
           label="patching: % logit gap restored")
ax[2].set_xlabel("top-k heads"); ax[2].set_ylabel("%"); ax[2].set_title("c  Heads → layer-16 direction", loc="left")
ax[2].legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(CTRL / "figs/controls.png", dpi=200); fig.savefig(CTRL / "figs/controls.pdf"); plt.show()

m = "mean"
lost = {t: SPEC[f"{t}/{m}"]["baseline_auc"] - SPEC[f"{t}/{m}"]["by_k"][40]["auc_top"] for t in TASKS}
rnd = SPEC[f"suicidality/{m}"]["baseline_auc"] - SPEC[f"suicidality/{m}"]["by_k"][40]["auc_rand_mean"]
print("VERDICT (mean-ablation, top-40):")
print(f"  suicidality AUC lost {lost['suicidality']:.3f} (random sets {rnd:.3f}); "
      f"sentiment {lost['sentiment']:.3f}; sports topic {lost['topic_sports']:.3f}")
print(f"  L16 projection separation left: {100 * WRITE['knockout'][m]['by_k'][40]['proj_gap_frac_remaining']:.0f}%;  "
      f"patching top-20 restores {100 * WRITE['patching'][20]['proj_ie_top']:.0f}% of the L16 projection")
summary = {"specificity": SPEC, "heads_write_direction": WRITE, "env": ENV}
(CTRL / "summary_controls.json").write_text(json.dumps(summary, indent=2, default=_js))
import shutil; shutil.make_archive(str(BASE / "revision_controls"), "zip", CTRL)
print("saved", CTRL / "summary_controls.json", "and", BASE / "revision_controls.zip")